## Generate Virginia pnode names and ids to subset PJM lmp data

1. Load sample PJM CSV: data/raw/usa_data/rt_hrl_lmps.csv
    - Find all unique pnode names and ids and subset these. Find also all unique zone values and unique type values

2. Load:
 - us states boundaries data: data/raw/usa_data/tl_2025_us_state/tl_2025_us_state.shp
 - electric substations data: data/raw/usa_data/electricSubstations/Shapefile/electric_substations_hifld_v4_FieldMods.shp
 - electric utility territories data: data/raw/usa_data/electricUtilityTerritories/Shapefile/Electric_Retail_Service_Territories_CustomFields.shp

3. Check which electric utilities have coverage within the state geographic boundaries of virginia. Use this to filter by zones in the PJM data later. 

4. Which substations are in virginia? Subset these substations. Filter out irrelevant ones (ie. unknown name), 

5. Match these substations to the pnodes in the PJM data by name. Use fuzzy matching and then use AI + manual checks to sort the rest 

- PJM pnode id $\neq$ HIFLD_ID

6. Merge. Then should have a file that includes(joined by the matched pnode names). 
    From substation data: HIFLD_Name; HIFLD_ID; city; zip; type; county; countyfips;StateShort;	StateFull; latitude; longitude
    From PJM data: pnode_id; pnode_name; type; zone

    - Summarise the PJM data type and zone unique values (used for data download later)

7. Then: subset this only to pnode_id

1. Load sample PJM CSV: data/raw/usa_data/rt_hrl_lmps.csv
    - Find all unique pnode names and ids and subset these. Find also all unique zone values and unique type values

In [31]:
#Set up file paths
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parents[1] / "src"))
from utils.paths import DATA_DIR

In [30]:
import pandas as pd

# Load the data
df = pd.read_csv("data/raw/usa_data/rt_hrl_lmps.csv")

# Inspect column names
print("Column names:")
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/usa_data/rt_hrl_lmps.csv'

In [27]:
#Check structure of hourly lmp data
df = pd.read_csv(DATA_DIR / "raw" / "usa_data" / "rt_hrl_lmps.csv")
df.head()

print(df.columns.tolist())

#Identify unique zone and type values
# Unique zone values
unique_zones = sorted(df["zone"].dropna().unique())

# Unique type values
unique_types = sorted(df["type"].dropna().unique())

print("Zones:", unique_zones)
print("Types:", unique_types)

['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_rt', 'total_lmp_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'row_is_current', 'version_nbr']
Zones: ['AECO', 'AEP', 'APS', 'ATSI', 'BGE', 'COMED', 'CPL', 'DAY', 'DEOK', 'DOM', 'DPL', 'DUKE', 'DUQ', 'EKPC', 'EXTERNAL', 'JCPL', 'METED', 'OVEC', 'PECO', 'PENELEC', 'PEPCO', 'PPL', 'PSEG', 'RECO']
Types: ['AGGREGATE', 'EHV', 'EXT', 'GEN', 'HUB', 'INTERFACE', 'LOAD', 'RESIDUAL_METERED_EDC', 'ZONE']


In [28]:
print(df.columns.tolist())

#Identify unique zone and type values
# Unique zone values
unique_zones = sorted(df["zone"].dropna().unique())

# Unique type values
unique_types = sorted(df["type"].dropna().unique())

print("Zones:", unique_zones)
print("Types:", unique_types)

['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_rt', 'total_lmp_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'row_is_current', 'version_nbr']
Zones: ['AECO', 'AEP', 'APS', 'ATSI', 'BGE', 'COMED', 'CPL', 'DAY', 'DEOK', 'DOM', 'DPL', 'DUKE', 'DUQ', 'EKPC', 'EXTERNAL', 'JCPL', 'METED', 'OVEC', 'PECO', 'PENELEC', 'PEPCO', 'PPL', 'PSEG', 'RECO']
Types: ['AGGREGATE', 'EHV', 'EXT', 'GEN', 'HUB', 'INTERFACE', 'LOAD', 'RESIDUAL_METERED_EDC', 'ZONE']


In [26]:
#Filter only by bus nodes (not aggregate or hub)
busnodes = df[df["type"].isin(["EHV", "GEN", "LOAD"])].copy()
print("Bus Type Nodes:", busnodes["type"].unique())

Bus Type Nodes: <ArrowStringArray>
['LOAD', 'GEN', 'EHV']
Length: 3, dtype: str


In [24]:
# Unique pnode names
unique_pnode_names = (
    busnodes["pnode_name"]
    .dropna()
    .unique()
)

# Optional: sort for readability
unique_pnode_names = sorted(unique_pnode_names)

print(unique_pnode_names[:10])  # preview
print("Number of unique pnode names:", len(unique_pnode_names))

['02AMSTED', '02ANGOLA', '02ARMCO', '02AYERSV', '02BRUSH', '02BRWELL', '02BWAK', '02CARDNT', '02CLINTN', '02COLLIN']
Number of unique pnode names: 6322
